In [2]:
!nvidia-smi

Wed Mar  4 11:45:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:B4:00.0 Off |                  Off |
| N/A   32C    P0             36W /  250W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
import os
print(os.getcwd())
os.chdir("/data/coding/ARC")

from dataclasses import dataclass
import os

@dataclass
class Config:
    data_root: str = 'data'
    split: str = 'evaluation'
    max_tasks: int = 0
    model_key: str = ''

    # SDPO LoRA 微调后的 adapter（来自 architect_sdpo_lora.ipynb）
    base_model_path: str = 'outputs/arc_lora_sft_C/lora_merged_model'
    sdpo_adapter_path: str = 'outputs/sdpo_lora_adapter_2'
    merged_model_path: str = 'outputs/sdpo_lora_merged_model_2'

    model_path: str = 'outputs/sdpo_lora_merged_model_2'
    tokenizer_path: str = 'outputs/sdpo_lora_merged_model_2'
    max_new_tokens: int = 931
    output_dir: str = 'outputs/architect_eval_TransAt1_sdpo2'
    viz_failures: bool = False

cfg = Config()

def get_truth(task, context):
    tests = task.get('test') or []
    if len(tests) != 1:
        return None
    return tests[0].get('output')

def get_task_id(task, context):
    return task.get('task_id', context.get('index'))


/data/coding


In [3]:
from pathlib import Path
import shutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from ARChitects.architect_transampling import build_architect_solver

merged_dir = Path(cfg.merged_model_path)
adapter_dir = Path(cfg.sdpo_adapter_path)
if not adapter_dir.exists():
    raise FileNotFoundError(f'SDPO adapter not found: {adapter_dir}')

def _merged_ready(path: Path) -> bool:
    if not path.exists():
        return False
    has_weights = any(
        (path / name).exists()
        for name in (
            'model.safetensors',
            'pytorch_model.bin',
            'pytorch_model-00001-of-00002.bin',
        )
    )
    return (path / 'config.json').exists() and has_weights

if not _merged_ready(merged_dir):
    if merged_dir.exists():
        print(f'[sdpo-eval] remove incomplete merged dir: {merged_dir}')
        shutil.rmtree(merged_dir)

    print(f'[sdpo-eval] merging adapter -> {merged_dir}')
    base = AutoModelForCausalLM.from_pretrained(
        cfg.base_model_path,
        device_map='auto',
        trust_remote_code=True,
    )
    lora = PeftModel.from_pretrained(base, str(adapter_dir), device_map='auto')
    merged = lora.merge_and_unload()

    merged_dir.mkdir(parents=True, exist_ok=True)
    try:
        merged.save_pretrained(
            str(merged_dir),
            safe_serialization=True,
            save_original_format=False,
        )
    except NotImplementedError as exc:
        print(f'[sdpo-eval] safetensors save failed, fallback to .bin: {exc}')
        merged.save_pretrained(
            str(merged_dir),
            safe_serialization=False,
            save_original_format=False,
        )

    tok = AutoTokenizer.from_pretrained(
        cfg.base_model_path,
        use_fast=False,
        trust_remote_code=True,
    )
    tok.save_pretrained(str(merged_dir))

    del base, lora, merged, tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f'[sdpo-eval] use existing merged model: {merged_dir}')

cfg.model_path = str(merged_dir)
cfg.tokenizer_path = str(merged_dir)

solver = build_architect_solver(
    model_key=cfg.model_key,
    model_path=cfg.model_path,
    tokenizer_path=cfg.tokenizer_path,
    max_new_tokens=cfg.max_new_tokens,
)


/data/miniconda/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[sdpo-eval] merging adapter -> outputs/sdpo_lora_merged_model_2


Loading weights: 100%|████████████████████████████████████████████████████████████████| 398/398 [00:01<00:00, 204.21it/s, Materializing param=model.norm.weight]
/data/miniconda/envs/torch/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Loading weights: 100%|████████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 473.81it/s, Materializing param=model.norm.weight]


[architect_beam] model moved to cuda


In [ ]:
from eval.core_C_hittops import ARCDataset, run_evaluation

dataset = ARCDataset(root=cfg.data_root, split=cfg.split, max_tasks=200, start_from=0)

reports = run_evaluation(
    dataset=dataset,
    solver=solver,
    output_dir=cfg.output_dir,
    model_id=cfg.model_path,
    model_key='architect_transampling_at1_sdpo',
    viz_failures=cfg.viz_failures,
    get_truth=get_truth,
    get_task_id=get_task_id,
)
print(reports['summary'])


[architect_beam] rollout 9/16 transform 9/16 grid key: 554481155815318558383158551184|44888811583173787883731585118 candidates so far: 8
[architect_beam] rollout 10/16 transform 10/16 grid key: 555448115851838888383158551184|48888115851738877883731585118 candidates so far: 9
[architect_beam] rollout 11/16 transform 11/16 grid key: 544811558325888353283158551184|48881158573122855823731585118 candidates so far: 10
[architect_beam] rollout 12/16 transform 12/16 grid key: 544811558832518331833158551184|48881158513872533283731585118 candidates so far: 11
[architect_beam] rollout 13/16 transform 13/16 grid key: 544811558538835855118445|488811585183381585118884|4852552213 candidates so far: 12
[architect_beam] rollout 14/16 transform 14/16 grid key: 544811558183838838318551184|488811585337388883735851188|4852 candidates so far: 13
[architect_beam] rollout 15/16 transform 15/16 grid key: 544811558513838448383158551184|48881158513738888883731585118 candidates so far: 14
[architect_beam] rollout

KeyboardInterrupt: 

[architect_beam] rollout 14/16 transform 14/16 grid key: 000800100040000|000800100040000|888888288828888|000800100040 candidates so far: 5
[architect_beam] rollout 15/16 transform 15/16 grid key: 000800100040000|000800100040000|888888288828888|000800100040 candidates so far: 6
[architect_beam] rollout 16/16 transform 16/16 grid key: 000800100040000|000800100040000|888888188828888|000800100040 candidates so far: 7
[architect_beam] Beam done, candidates: 8
[architect_beam] candidate 1 mean_logprob=-0.68 min=-2.52 max=-0.19
[architect_beam] candidate 2 mean_logprob=-6.39 min=-10.32 max=-2.32
[architect_beam] candidate 3 mean_logprob=-7.18 min=-12.29 max=-3.10
[architect_beam] candidate 4 mean_logprob=-11.34 min=-24.34 max=-2.61
[architect_beam] candidate 5 mean_logprob=-9.46 min=-12.22 max=-4.66
[architect_beam] candidate 6 mean_logprob=-3.35 min=-9.17 max=-0.81
[architect_beam] candidate 7 mean_logprob=-17.28 min=-25.24 max=-4.74
[architect_beam] candidate 8 mean_logprob=-13.56 min=-17.4